In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [16]:
df= pd.read_csv("/content/popular_quotes.csv")

In [17]:
df.head()

,quotes,tags
0,Be yourself; everyone else is already taken.,"['attributed-no-source', 'be-yourself', 'hones..."
1,"I'm selfish, impatient and a little insecure. ...","['attributed-no-source', 'best', 'life', 'love..."
2,Two things are infinite: the universe and huma...,"['attributed-no-source', 'human-nature', 'humo..."
3,"So many books, so little time.","['books', 'humor']"
4,A room without books is like a body without a ...,"['attributed-no-source', 'books', 'simile', 's..."


In [18]:
df['quotes'][0]

'Be yourself; everyone else is already taken.'

In [19]:
df.shape

(2481, 2)

In [20]:
quotes= df['quotes']
quotes.head()


,quotes
0,Be yourself; everyone else is already taken.
1,"I'm selfish, impatient and a little insecure. ..."
2,Two things are infinite: the universe and huma...
3,"So many books, so little time."
4,A room without books is like a body without a ...


In [21]:
quotes.str.lower()

,quotes
0,be yourself; everyone else is already taken.
1,"i'm selfish, impatient and a little insecure. ..."
2,two things are infinite: the universe and huma...
3,"so many books, so little time."
4,a room without books is like a body without a ...
...,...
2476,"if you love two people at the same time, choos..."
2477,if he loved with all the powers of his puny be...
2478,to hurt is as human as to breathe.
2479,the present changes the past. looking back you...


In [22]:
import string
translator= str.maketrans('','',string.punctuation)
quotes=quotes.apply(lambda x: x.translate(translator))
quotes.str.lower()

,quotes
0,be yourself everyone else is already taken
1,im selfish impatient and a little insecure i m...
2,two things are infinite the universe and human...
3,so many books so little time
4,a room without books is like a body without a ...
...,...
2476,if you love two people at the same time choose...
2477,if he loved with all the powers of his puny be...
2478,to hurt is as human as to breathe
2479,the present changes the past looking back you ...


In [23]:
quotes.head()
quotes.str.lower()

,quotes
0,be yourself everyone else is already taken
1,im selfish impatient and a little insecure i m...
2,two things are infinite the universe and human...
3,so many books so little time
4,a room without books is like a body without a ...
...,...
2476,if you love two people at the same time choose...
2477,if he loved with all the powers of his puny be...
2478,to hurt is as human as to breathe
2479,the present changes the past looking back you ...


In [24]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [25]:
vocab_size= 10000
tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [26]:
word_index=tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

7271


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [27]:
sequence= tokenizer.texts_to_sequences(quotes)

In [28]:
quotes[0]

'Be yourself everyone else is already taken'

In [29]:
sequence[0]

[12, 111, 244, 183, 7, 402, 647]

In [30]:
x = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
   input_seq = seq[:i]
   output_seq= seq[i]
   x.append(input_seq)
   y.append(output_seq)

In [31]:
len(x)

73125

In [32]:
len(y)

73125

In [33]:
max_len = max(len(item) for item in x)
print(max_len)

748


In [34]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded= pad_sequences(x,maxlen=max_len,padding='pre')

In [35]:
X_padded

array([[  0,   0,   0, ...,   0,   0,  12],
       [  0,   0,   0, ...,   0,  12, 111],
       [  0,   0,   0, ...,  12, 111, 244],
       ...,
       [  0,   0,   0, ...,  16,  26,   2],
       [  0,   0,   0, ...,  26,   2, 140],
       [  0,   0,   0, ...,   2, 140,  16]], dtype=int32)

In [36]:
y= np.array(y)

In [37]:
X_padded.shape

(73125, 748)

In [38]:
y.shape

(73125,)

In [39]:
from tensorflow.keras.utils import to_categorical

# One-hot encode the target labels 'y'
y_encoded = to_categorical(y, num_classes=vocab_size)

print("Original y shape:", y.shape)
print("One-hot encoded y shape:", y_encoded.shape)

Original y shape: (73125,)
One-hot encoded y shape: (73125, 10000)


In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , LSTM , Dense,SimpleRNN

In [41]:
embedding_dim=50
rnn_units= 128

In [42]:
rnn_model= Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

In [43]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [44]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [45]:
lstm_model= Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim,input_length=max_len)
)

In [46]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [47]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [48]:
epochs= 10
batch_size= 128

In [49]:
history_rnn = rnn_model.fit(
    X_padded, y_encoded,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 44s 84ms/step - accuracy: 0.0423 - loss: 6.7068 - val_accuracy: 0.0559 - val_loss: 6.5510
Epoch 2/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - accuracy: 0.0647 - loss: 6.1745 - val_accuracy: 0.0777 - val_loss: 6.4082
Epoch 3/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - accuracy: 0.0925 - loss: 5.8592 - val_accuracy: 0.0931 - val_loss: 6.2962
Epoch 4/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 41s 72ms/step - accuracy: 0.1086 - loss: 5.5801 - val_accuracy: 0.1004 - val_loss: 6.2919
Epoch 5/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - accuracy: 0.1208 - loss: 5.3362 - val_accuracy: 0.1058 - val_loss: 6.3185
Epoch 6/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 41s 72ms/step - accuracy: 0.1322 - loss: 5.1130 - val_accuracy: 0.1102 - val_loss: 6.3307
Epoch 7/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - accuracy: 0.1455 - loss: 4.9077 - val_accuracy: 0.1089 - val_loss: 6.3836
Epoch 8/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 72ms/step - accuracy: 0.1569 - loss: 4.7167 - 

In [50]:
lstm_model.save("lstm_model.h5")

In [51]:
index_to_word={}
for word,index in word_index.items():
  index_to_word[index]= word

In [52]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'with',
 21: 'if',
 22: 'what',
 23: 'love',
 24: 'all',
 25: 'can',
 26: 'when',
 27: 'my',
 28: 'do',
 29: 'he',
 30: 'will',
 31: 'me',
 32: 'as',
 33: 'who',
 34: 'they',
 35: 'was',
 36: 'or',
 37: 'life',
 38: 'one',
 39: 'like',
 40: 'there',
 41: 'never',
 42: 'on',
 43: 'its',
 44: 'people',
 45: 'at',
 46: 'so',
 47: 'no',
 48: 'dont',
 49: 'them',
 50: 'more',
 51: 'just',
 52: 'his',
 53: 'than',
 54: 'this',
 55: 'know',
 56: 'only',
 57: 'because',
 58: 'want',
 59: 'how',
 60: 'world',
 61: 'up',
 62: 'time',
 63: 'things',
 64: 'by',
 65: 'said',
 66: 'from',
 67: 'think',
 68: 'about',
 69: 'us',
 70: 'always',
 71: 'our',
 72: 'an',
 73: 'make',
 74: 'out',
 75: 'good',
 76: 'go',
 77: 'way',
 78: 'has',
 79: 'were',
 80: 'am',
 81: 'live',
 82: 'something

In [53]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [54]:
def predictor(model,tokenizer,text,max_len):
  text= text.lower()
  sequence= tokenizer.texts_to_sequences([text])[0]
  padded_seq= pad_sequences([sequence],maxlen=max_len,padding='pre')
  pred= model.predict(seq,verbose=0)
  pred_index= np.argmax(pred)
  return index_to_word[pred_index,""]


In [55]:
def predictor(model, tokenizer, text, max_len):
  text = text.lower()
  sequence = tokenizer.texts_to_sequences([text])[0]
  padded_seq = pad_sequences([sequence], maxlen=max_len, padding='pre')
  pred = model.predict(padded_seq, verbose=0)
  pred_index = np.argmax(pred)
  return index_to_word.get(pred_index, "")

seed_text = "Be Yourself"
next_word = predictor(lstm_model, tokenizer, seed_text, max_len)
print(seed_text, next_word)

Be Yourself 


In [56]:
def generate_text(model, tokenizer, seed_text, max_len, num_words):
  for _ in range(num_words):
    next_word = predictor(model, tokenizer, seed_text, max_len)
    seed_text += " " + next_word
  return seed_text

In [57]:
seed = "the meaning of life"
generated_result = generate_text(lstm_model, tokenizer, seed, max_len, 10)
print(generated_result)

the meaning of life          


In [58]:
import pickle
with open ("tokenizer.pkl","wb") as f:
  pickle.dump(tokenizer,f)

In [59]:
with open("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)

In [60]:
!pip install -q gradio

In [61]:
import gradio as gr

def ui_predict(seed_text, num_words, model_choice):
    # Select the model based on user choice
    selected_model = rnn_model if model_choice == "RNN Model" else lstm_model

    # Generate text
    result = generate_text(selected_model, tokenizer, seed_text, max_len, int(num_words))
    return result

# Define a beautiful modern theme and interface layout
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🖋️ Quotes & Text Generator AI
        Generate custom text continuations based on popular quotes using your trained Recurrent Neural Networks.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            seed_input = gr.Textbox(
                label="Seed Text (Starting Phrase)",
                placeholder="Type something like 'Be yourself' or 'The meaning of life'...",
                value="Be yourself"
            )
            word_slider = gr.Slider(
                minimum=1,
                maximum=30,
                value=10,
                step=1,
                label="Number of Words to Generate"
            )
            model_dropdown = gr.Dropdown(
                choices=["RNN Model", "LSTM Model"],
                value="RNN Model",
                label="Select Model (Note: RNN is fully trained)"
            )
            submit_btn = gr.Button("✨ Generate Text", variant="primary")

        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label="Generated Continuation Output",
                interactive=False,
                lines=6
            )

    # Connect button click and inputs to the prediction logic
    submit_btn.click(
        fn=ui_predict,
        inputs=[seed_input, word_slider, model_dropdown],
        outputs=output_text
    )

# Launch the app inline inside Colab
demo.launch(quiet=True)

* Running on public URL: https://d0afb44abec2ca8303.gradio.live
